# 05 — Execution Eval Baseline

## What this notebook does

Retrieves KernelPack code chunks for a given eval task, assembles a grounded
generation prompt, and inspects the agent's output against the ground-truth
checker.

## Handrolled vs MCP

| Step | This notebook (handrolled) | Final MCP pipeline |
|------|---------------------------|--------------------|
| Query formulation | Raw prompt text used verbatim as retrieval query | Agent generates its own `retrieve_code()` call |
| Prompt assembly | Manual string construction in this cell | Structured tool input/output |
| Agent invocation | Human copy-pastes prompt into agent | Codex calls `retrieve_code()` and generates inline |
| Checker invocation | Handrolled `subprocess` call below | Sandboxed execution tool |

In [2]:
import subprocess
from pathlib import Path
from IPython.display import Markdown, display
from qdrant_client import QdrantClient
from kernelpack_rag.config import COLLECTIONS_CONFIG
from kernelpack_rag.ingest import CODE_COLLECTION
from kernelpack_rag.embed.jinacode import JinaCodeEmbedder
from kernelpack_rag.retrieve import hybrid, Candidate

TASK_TIER = "easy"  # change to "medium" or "hard" to rerun

RAGSYSTEM_DIR = Path("../eval/ragsystem").resolve()
TIER_MAP = {
    "easy":   {"prompt": "prompt_easy.txt",   "driver": "easy_scalar_c4_matern_driver.py"},
    "medium": {"prompt": "prompt_medium.txt",  "driver": "medium_df_c4_matern_driver.py"},
    "hard":   {"prompt": "prompt_hard.txt",    "driver": "hard_local_dfphs_poly_driver.py"},
}
# PROMPT_FILE = RAGSYSTEM_DIR / TIER_MAP[TASK_TIER]["prompt"]
PROMPT_FILE = RAGSYSTEM_DIR / f"modified_prompt_easy.txt"
DRIVER_FILE = RAGSYSTEM_DIR / TIER_MAP[TASK_TIER]["driver"]
OUTPUT_DIR  = RAGSYSTEM_DIR / "outputs" / TASK_TIER
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# QUERY = PROMPT_FILE.read_text()
QUERY = (RAGSYSTEM_DIR / "modified_prompt_easy.txt").read_text()

client   = QdrantClient(host="localhost", port=6333)
embedder = JinaCodeEmbedder()

print(f"Tier:       {TASK_TIER}")
print(f"Prompt:     {PROMPT_FILE}")
print(f"Driver:     {DRIVER_FILE}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"Collection: {CODE_COLLECTION}")
print(f"Collections in config: {list(COLLECTIONS_CONFIG)}")

/Users/jordanchambers/public-projects/scientific-codebase-rag/.venv/lib/python3.12/site-packages/qdrant_client/qdrant_remote.py:290: UserWarning: Failed to obtain server version. Unable to check client-server compatibility. Set check_compatibility=False to skip version check.
  show_warning(
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 28662.44it/s]

Tier:       easy
Prompt:     /Users/jordanchambers/public-projects/scientific-codebase-rag/codebase-rag/eval/ragsystem/modified_prompt_easy.txt
Driver:     /Users/jordanchambers/public-projects/scientific-codebase-rag/codebase-rag/eval/ragsystem/easy_scalar_c4_matern_driver.py
Output dir: /Users/jordanchambers/public-projects/scientific-codebase-rag/codebase-rag/eval/ragsystem/outputs/easy
Collection: kernelpack_code
Collections in config: ['kernelpack_code', 'kernelpack_papers']


---
## Section 1 — RAG Pipeline

### Step 1 — Retrieve chunks

Uses the `hybrid` plan: dense (`ctx__jinacode`, dim=896) + BM25 sparse (`bm25_code`),
fused with RRF. The full prompt text is the retrieval query.

In [4]:
K = 10
candidates = hybrid(
    QUERY,
    client=client,
    collection=CODE_COLLECTION,
    embedder=embedder,
    k=K,
)

print(f"Retrieved {len(candidates)} chunks\n")
for c in candidates:
    preview = str(c.payload.get("text") or "")[:120].replace("\n", " ")
    print(f"[rank {c.fused_rank}] id={c.point_id}  score={c.fused_score:.4f}")
    print(f"  {preview}")
    print()

Retrieved 10 chunks

[rank 0] id=127db2ec-a3e3-5bec-b1b9-981ddb8af077  score=0.5000
  def dfc4_blocks_periodic_3d_z(     x: np.ndarray,     y: np.ndarray,     epsilon: float, ) -> tuple[np.ndarray, np.ndarr

[rank 1] id=85ab83fe-13ad-5f7c-bcf9-4643097889bb  score=0.5000
  @dataclass class RBFStencil:     a: np.ndarray = field(default_factory=lambda: np.zeros((0, 0)))     solve_lhs: np.ndarr

[rank 2] id=5199e3b2-01bc-55b8-8be2-b9152d7a6e6c  score=0.3333
      def build_closed_geometric_model_ps(self, dim: int, rad: float, nb: int, ne: int | None = None, method: int = 1, sup

[rank 3] id=2b457a6f-6630-5641-8767-49989059bbc9  score=0.3333
          if neu_coeff != 0:             for d in range(self.s_dim):                 diff = x_subset[:, d : d + 1] - x[Non

[rank 4] id=c24f5e93-cdcc-5abc-bf5a-0bd858543ea8  score=0.2500
          if name in {"interp", "interpolation"}:             return self.interp_op(sp, op, r_rhs, x_subset, x, x_at_origi

[rank 5] id=8a8c5ea3-7a24-5410-a66e-05126141

### Step 2 — Inspect retrieved chunks

Full chunk text and payload metadata are displayed below. **This is a manual
inspection gate.** If the chunks look off-topic or missing key symbols, stop here
and adjust the query or switch retrieval plans before spending a generation call.
The quality of what the agent can produce is bounded by what is in these chunks.

In [5]:
for c in candidates:
    print(f"=== rank {c.fused_rank}  id={c.point_id}  score={c.fused_score:.4f} ===")
    for k, v in c.payload.items():
        if k != "text":
            print(f"  {k}: {v}")
    print()
    print(c.payload.get("text", "(no text)"))
    print()

=== rank 0  id=127db2ec-a3e3-5bec-b1b9-981ddb8af077  score=0.5000 ===
  context_header: # import numpy as np
# above: def dfc4_blocks_periodic_2d(x: np.ndarray, y: np.ndarray, epsilon: float) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
  llm_summary: This function computes six 3D radial basis function finite difference (RBF-FD) differentiation matrices or blocks with periodic boundary conditions applied along the z-dimension, using a Gaussian-like kernel modulated by trigonometric terms to enforce periodicity. The key inputs are coordinate arrays x and y representing node locations in 3D space, and epsilon, a shape parameter controlling the kernel's width and smoothness. It fits into the RBF-FD solver workflow by providing the necessary differentiation weights for approximating derivatives in PDEs on domains with periodicity in the z-direction, enabling accurate and efficient discretization of differential operators under these conditions.
  chunk_type: function
  module: kernelpack.

### Step 3 — Assemble generation prompt

The task prompt and retrieved chunks are combined into a single generation prompt.
The system instruction tells the agent to use **only** the provided context and
to produce two output files:

1. `ragcode_{tier}.py` — the solution script
2. `generation_trace_{tier}.md` — a chronological reasoning log

Both files go to `eval/ragsystem/outputs/{tier}/`.

In [6]:
ragcode_filename = f"ragcode_{TASK_TIER}.py"
trace_filename   = f"generation_trace_{TASK_TIER}.md"
output_dir_str   = f"eval/ragsystem/outputs/{TASK_TIER}"

chunk_block = "\n\n---\n\n".join(
    f"# Chunk {i + 1}  (id={c.point_id}  score={c.fused_score:.4f})\n"
    + str(c.payload.get("text") or "")
    for i, c in enumerate(candidates)
)

generation_prompt = f"""\
You are a scientific computing assistant.

Use ONLY the code chunks provided below as implementation context. Do not invent
functions, signatures, or APIs that are not present in the retrieved chunks.

## Task

{QUERY}

## Retrieved Code Context

{chunk_block}

## Required Outputs

Write both of the following files to `{output_dir_str}/`:

1. `{ragcode_filename}` — a standalone Python driver script that fulfills the
   task above. It must be self-contained and runnable with `python {ragcode_filename}`.

2. `{trace_filename}` — a chronological reasoning log. For each decision step
   record:
   - which chunk / file / symbol you referenced
   - why you chose it
   - whether it was ultimately useful
   - what (if anything) from it ended up in `{ragcode_filename}`
   - suggested `math_terms` and `api_terms` metadata that would help a future
     RAG system retrieve this evidence
"""

print("=" * 72)
print("COPY EVERYTHING BELOW THIS LINE")
print("=" * 72)
print(generation_prompt)
print("=" * 72)
print("COPY EVERYTHING ABOVE THIS LINE")
print("=" * 72)

COPY EVERYTHING BELOW THIS LINE
You are a scientific computing assistant.

Use ONLY the code chunks provided below as implementation context. Do not invent
functions, signatures, or APIs that are not present in the retrieved chunks.

## Task

Write a standalone Python driver script that performs scalar field interpolation
using a scalar-valued radial basis function. Make sure to use proper NumPy
techniques to keep the implementation efficient, and it is fine to use SciPy
and Matplotlib.

You have been provided with relevant code embeddings for this task. Use those
code embeddings as supporting implementation context when constructing the
driver, especially for the logic structure, kernel assembly style, experiment
organization, and plotting flow, while still producing a single self-contained
script.

The script should:

1. Define a smooth synthetic scalar target field on a 2D square domain:
   f(x, y) = sin(2*pi*x)*cos(3*pi*y) + 0.2*exp(-5*((x-0.25)^2 + (y+0.10)^2)).
2. Use the square 

### Step 4 — Handoff

1. **Copy** the prompt printed above.
2. **Paste** it into your agent (Claude Code, Codex, GPT-4.1, etc.).
3. The agent should write both output files to `eval/ragsystem/outputs/{TASK_TIER}/`:
   - `ragcode_{TASK_TIER}.py`
   - `generation_trace_{TASK_TIER}.md`
4. **Come back and run Section 2** once the agent has finished.

---
## Section 2 — Generation Quality Inspection

### Step 5 — Run checker

`checker.py` runs both scripts in isolated temp directories (via `RAGSYSTEM_OUTPUT_DIR`),
locates each script's `*_errors.txt` output file, and compares the ablation rows
numerically with `atol=1e-8`, `rtol=1e-6`.

In [6]:
ragcode_path = OUTPUT_DIR / f"ragcode_{TASK_TIER}.py"
driver_path  = DRIVER_FILE

result = subprocess.run(
    ["python", "checker.py", str(ragcode_path), str(driver_path)],
    capture_output=True,
    text=True,
    cwd=str(RAGSYSTEM_DIR),
)

print("--- stdout ---")
print(result.stdout or "(empty)")
print("--- stderr ---")
print(result.stderr or "(empty)")
print(f"Return code: {result.returncode}")

--- stdout ---
N epsilon rel_l2
   50        1 4.44107229e+00
   50        2 2.31212231e+00
   50        4 1.15969788e+00
   50        8 1.00718450e+00
  100        1 4.13682121e+00
  100        2 2.37543318e+00
  100        4 1.21123486e+00
  100        8 1.01227305e+00
  500        1 1.29026185e+00
  500        2 1.10009488e+00
  500        4 1.10982313e+00
  500        8 1.03512506e+00
 1000        1 1.64959340e+00
 1000        2 1.14757385e+00
 1000        4 8.85887132e-01
 1000        8 9.64350232e-01
 2000        1 2.36445312e-01
 2000        2 2.36590296e-01
 2000        4 2.37549759e-01
 2000        8 2.20171521e-01
Wrote outputs to: /var/folders/vt/7d703f_x4gn3vcd1ty8pcsc80000gn/T/ragcheck_a_oji2j3bl
N=  50, epsilon= 0.75, relL2=1.382e+00
N=  50, epsilon= 1.50, relL2=1.330e+00
N=  50, epsilon= 3.00, relL2=1.230e+00
N= 100, epsilon= 0.75, relL2=4.906e-01
N= 100, epsilon= 1.50, relL2=4.734e-01
N= 100, epsilon= 3.00, relL2=4.516e-01
N= 500, epsilon= 0.75, relL2=2.788e-02
N= 500, 

### Step 6 — Display generation trace

In [7]:
trace_path = OUTPUT_DIR / f"generation_trace_{TASK_TIER}.md"
if trace_path.exists():
    display(Markdown(trace_path.read_text()))
else:
    print("generation_trace not found — the agent did not produce it.")

# Generation Trace: `ragcode_easy.py`

This trace records the chronological implementation decisions used to produce
`ragcode_easy.py` from the provided prompt and retrieved code chunks.

## Step 1: Scope the deliverable as a standalone driver

- Referenced chunk / file / symbol: `easy_task_prompt.md`, `Task`, `Required Outputs`
- Why chosen: The prompt defines the required scalar interpolation experiment, output filenames, output directory behavior, and trace requirements.
- Ultimately useful: Yes.
- What ended up in `ragcode_easy.py`: A single executable driver script with constants for the square domain, seed, node counts, evaluation count, kernel shape parameters, output files, printed rows, and plot generation.
- suggested `math_terms`: `scalar field interpolation`, `2D square domain`, `relative L2 error`, `ablation study`
- suggested `api_terms`: `__main__`, `os.environ`, `pathlib.Path`, `RAGSYSTEM_OUTPUT_DIR`

## Step 2: Implement the synthetic target field with vectorized NumPy operations

- Referenced chunk / file / symbol: `easy_task_prompt.md`, target formula `f(x, y) = sin(2*pi*x)*cos(3*pi*y) + 0.2*exp(-5*((x-0.25)^2 + (y+0.10)^2))`
- Why chosen: This formula is the scalar field the interpolant must reproduce on `[-1, 1] x [-1, 1]`.
- Ultimately useful: Yes.
- What ended up in `ragcode_easy.py`: `target_field(points)` converts input to `np.asarray`, extracts `x` and `y` columns, and evaluates the oscillatory term plus Gaussian bump without Python loops.
- suggested `math_terms`: `trigonometric scalar field`, `Gaussian bump`, `smooth synthetic function`, `vectorized evaluation`
- suggested `api_terms`: `np.asarray`, `np.sin`, `np.cos`, `np.exp`, `np.pi`

## Step 3: Choose vectorized pairwise kernel assembly

- Referenced chunk / file / symbol: Chunk 1, `dfc4_blocks_periodic_3d_z`, symbols `np.asarray`, `diff = x[:, None, :] - y[None, :, :]`, `r = np.sqrt(...)`, `exp_term = np.exp(-eps * r)`
- Why chosen: Although the chunk is for derivative blocks in a periodic 3D setting, it shows the local style for dense kernel/block assembly: coerce arrays to floats, build pairwise distances in one array operation, compute powers of `epsilon`, and multiply by an exponential radial factor.
- Ultimately useful: Yes.
- What ended up in `ragcode_easy.py`: `matern_c4_kernel(x, y, epsilon)` builds a dense pairwise distance matrix with SciPy's vectorized `cdist`, computes `er = epsilon * r`, and returns the scalar C4 Matern kernel matrix `exp(-er) * (1 + er + er**2 / 3)`.
- suggested `math_terms`: `C4 Matern kernel`, `radial basis function`, `pairwise Euclidean distance`, `shape parameter`, `exponential kernel`
- suggested `api_terms`: `scipy.spatial.distance.cdist`, `np.asarray`, `np.exp`, `ndarray broadcasting`

## Step 4: Use a global interpolation matrix and coefficient solve

- Referenced chunk / file / symbol: Chunk 2, `RBFStencil`, fields `a`, `solve_lhs`, `coeffs`, `x_stencil`, method `get_interp_mat`
- Why chosen: This chunk establishes that RBF interpolation is represented through a dense interpolation matrix and solved coefficients/weights, even though the task needs a global scalar driver rather than a stencil class.
- Ultimately useful: Yes.
- What ended up in `ragcode_easy.py`: `solve_interpolant(nodes, values, epsilon)` assembles `K(nodes, nodes)` and solves for interpolation weights. The code keeps the interpolation matrix local instead of introducing a class, because the requested output is a self-contained script.
- suggested `math_terms`: `interpolation matrix`, `RBF coefficients`, `global interpolant`, `dense linear solve`
- suggested `api_terms`: `scipy.linalg.cho_factor`, `scipy.linalg.cho_solve`, `np.linalg.solve`, `np.eye`

## Step 5: Add tiny regularization only as a numerical fallback

- Referenced chunk / file / symbol: Chunk 4, `build_closed_geometric_model_ps`, symbols `k = phs_kernel(...)`, `reg = 1e-12 * max(1.0, np.abs(k).max(initial=0.0))`, `np.linalg.solve(k + reg * np.eye(...), data)`
- Why chosen: This code shows the retrieved codebase's pattern for solving RBF-like systems with a small scale-aware diagonal regularization term.
- Ultimately useful: Yes.
- What ended up in `ragcode_easy.py`: `solve_interpolant` first tries the exact kernel matrix with Cholesky. If that fails numerically, it retries with small scale-aware jitter and finally falls back to `np.linalg.solve` with a small diagonal term.
- suggested `math_terms`: `regularization`, `diagonal jitter`, `positive definite kernel`, `condition number`, `Cholesky factorization`
- suggested `api_terms`: `np.abs`, `ndarray.max`, `np.eye`, `cho_factor`, `cho_solve`, `np.linalg.LinAlgError`

## Step 6: Generate reproducible distinct interpolation and evaluation points

- Referenced chunk / file / symbol: Chunk 3, returned options keys `seed`, `base_seed`, `deterministic`
- Why chosen: This chunk is about sampler option normalization rather than interpolation, but it emphasizes explicit seed handling and deterministic reproducibility.
- Ultimately useful: Partially.
- What ended up in `ragcode_easy.py`: `RANDOM_SEED = 20260617`, `np.random.default_rng(RANDOM_SEED)`, and `uniform_distinct_points(...)`. The script draws one unique pool of `max(NODE_COUNTS) + 500` points, uses the first part as the interpolation pool, and reserves exactly 500 separate evaluation points.
- suggested `math_terms`: `uniform random sampling`, `distinct interpolation nodes`, `evaluation set`, `reproducibility`
- suggested `api_terms`: `np.random.default_rng`, `Generator.uniform`, `np.unique`, `np.vstack`

## Step 7: Organize the ablation over node count and shape parameter

- Referenced chunk / file / symbol: `easy_task_prompt.md`, node counts `[50, 100, 500, 1000, 2000]`; Chunk 4 symbols `ntarget`, `ns`, sampled/evaluation set organization
- Why chosen: The prompt mandates the node-count sweep, while Chunk 4 provides a useful pattern of building a source sample set and an evaluation sample set separately.
- Ultimately useful: Yes.
- What ended up in `ragcode_easy.py`: Nested loops over `NODE_COUNTS` and `EPSILONS`, with `nodes = interpolation_pool[:node_count]` for nested ablation samples and one fixed evaluation set for every configuration.
- suggested `math_terms`: `parameter sweep`, `node-count ablation`, `kernel shape parameter`, `fixed test set`
- suggested `api_terms`: `for loop`, `list[tuple[int, float, float]]`, `array slicing`

## Step 8: Compute and print relative L2 errors

- Referenced chunk / file / symbol: `easy_task_prompt.md`, requirements to evaluate on the test set, compute relative L2 error, and print results
- Why chosen: This is the core experiment output.
- Ultimately useful: Yes.
- What ended up in `ragcode_easy.py`: `relative_l2(predicted, truth)` uses `np.linalg.norm(predicted - truth) / np.linalg.norm(truth)`, and `main()` prints rows headed by `N epsilon rel_l2`.
- suggested `math_terms`: `relative L2 norm`, `prediction error`, `test-set evaluation`
- suggested `api_terms`: `np.linalg.norm`, `print`, `f-string formatting`

## Step 9: Write the numeric ablation table

- Referenced chunk / file / symbol: `easy_task_prompt.md`, output row format `N epsilon rel_l2`
- Why chosen: The prompt specifies a plain text file with a comment header and one numeric row per configuration.
- Ultimately useful: Yes.
- What ended up in `ragcode_easy.py`: `scalar_rbf_matern_c4_errors.txt` is written under `RAGSYSTEM_OUTPUT_DIR` when set, otherwise beside the script in `rbf_outputs/`. The first line is `# N epsilon rel_l2`; all data lines use numeric row format.
- suggested `math_terms`: `ablation table`, `configuration error`, `numeric experiment output`
- suggested `api_terms`: `Path.open`, `write`, `encoding='utf-8'`, `os.environ.get`

## Step 10: Produce an ablation plot

- Referenced chunk / file / symbol: `easy_task_prompt.md`, plot requirement; Chunk 4 organization of repeated sample/evaluation outputs
- Why chosen: The prompt requires at least one plot summarizing ablation results.
- Ultimately useful: Yes.
- What ended up in `ragcode_easy.py`: `plot_ablation(...)` writes `scalar_rbf_matern_c4_ablation.svg`, a log-log SVG plot of relative L2 error versus interpolation node count for each epsilon.
- suggested `math_terms`: `log-log error plot`, `convergence trend`, `shape-parameter comparison`
- suggested `api_terms`: `Path.write_text`, `SVG path`, `np.log10`, `HTML escaping`

## Step 11: Produce a qualitative field/error plot

- Referenced chunk / file / symbol: `easy_task_prompt.md`, requirement for true scalar field, predicted scalar field, and pointwise error; Chunk 1 radial kernel evaluation style
- Why chosen: The prompt requires a representative qualitative visualization, and Chunk 1 supports evaluating dense blocks between evaluation points and interpolation centers.
- Ultimately useful: Yes.
- What ended up in `ragcode_easy.py`: `plot_qualitative(...)` evaluates the representative interpolant on a regular grid and writes `scalar_rbf_matern_c4_qualitative.svg` with three panels: true field, predicted field, and pointwise error.
- suggested `math_terms`: `pointwise error`, `regular evaluation grid`, `scalar field visualization`, `representative configuration`
- suggested `api_terms`: `np.linspace`, `np.meshgrid`, `np.column_stack`, `reshape`, `base64.b64encode`, `zlib.compress`

## Step 12: Avoid unavailable plotting dependencies while keeping the script self-contained

- Referenced chunk / file / symbol: `easy_task_prompt.md`, "it is fine to use SciPy and Matplotlib"
- Why chosen: Matplotlib is allowed but not required by the wording. The local verification environment had NumPy and SciPy but not Matplotlib.
- Ultimately useful: Yes.
- What ended up in `ragcode_easy.py`: The script uses SciPy for distances and Cholesky solves, but generates SVG plots directly using the Python standard library. Heatmaps are embedded as base64 PNG images generated from NumPy arrays, so no Matplotlib dependency is needed.
- suggested `math_terms`: `heatmap`, `color mapping`, `plot serialization`
- suggested `api_terms`: `base64`, `struct.pack`, `zlib.crc32`, `Path.write_text`, `SVG image href`

